**Chapter 19 – Training and Deploying TensorFlow Models at Scale**

_This notebook contains all the sample code and solutions to the exercises in chapter 19._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/19_training_and_deploying_at_scale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/19_training_and_deploying_at_scale.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.7 or above:

In [1]:
import sys

assert sys.version_info >= (3, 7)

**Warning**: the latest TensorFlow versions are based on Keras 3. For chapters 10-15, it wasn't too hard to update the code to support Keras 3, but unfortunately it's much harder for this chapter, so I've had to revert to Keras 2. To do that, I set the `TF_USE_LEGACY_KERAS` environment variable to `"1"` and import the `tf_keras` package. This ensures that `tf.keras` points to `tf_keras`, which is Keras 2.*.

In [2]:
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    import os
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    import tf_keras

And TensorFlow ≥ 2.8:

In [3]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

I0000 00:00:1782149392.176439     448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782149392.454821     448 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782149394.267010     448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


If running on Colab or Kaggle, you need to install the Google AI Platform client library, which will be used later in this notebook. You can ignore the warnings about version incompatibilities.

* **Warning**: On Colab, you must restart the Runtime after the installation, and continue with the next cells.

In [4]:
import sys
if "google.colab" in sys.modules or "kaggle_secrets" in sys.modules:
    %pip install -q -U google-cloud-aiplatform

This chapter discusses how to run or train a model on one or more GPUs, so let's make sure there's at least one, or else issue a warning:

In [5]:
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if "kaggle_secrets" in sys.modules:
        print("Go to Settings > Accelerator and select GPU.")

# Serving a TensorFlow Model

Let's start by deploying a model using TF Serving, then we'll deploy to Google Vertex AI.

## Using TensorFlow Serving

The first thing we need to do is to build and train a model, and export it to the SavedModel format.

### Exporting SavedModels

Let's load the MNIST dataset, scale it, and split it.

In [6]:
from pathlib import Path
import tensorflow as tf

# extra code – load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# extra code – build & train an MNIST model (also handles image preprocessing)
tf.random.set_seed(42)
tf.keras.backend.clear_session()
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8),
    tf.keras.layers.Rescaling(scale=1 / 255),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

# Lo que hace SavedModel es guardar el modelo completo en un formato estándar que TensorFlow Serving, 
# Vertex AI o cualquier aplicación TensorFlow pueden cargar después.
model_name = "my_mnist_model"
model_version = "0001" # TensorFlow Serving trabaja con versiones.
model_path = Path(model_name) / model_version # produce my_mnist_model/0001. En Windows -> my_mnist_model\0001
model.export(model_path) # generar un SavedModel para TensorFlow Serving

/home/srmjf/tf/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1782149396.871038     448 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9507 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Epoch 1/10


I0000 00:00:1782149397.993559     565 service.cc:153] XLA service 0x76a49803f2d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782149397.993599     565 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1782149398.002733     565 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782149398.045069     565 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1782149398.053277     565 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_508__.12
I0000 00:00:1782149398.494301     565 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set in

  94/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.2205 - loss: 2.1977

I0000 00:00:1782149399.506563     565 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1697/1719 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7111 - loss: 1.1100

I0000 00:00:1782149402.692731     565 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_508__.12


1719/1719 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8269 - loss: 0.6911 - val_accuracy: 0.9018 - val_loss: 0.3678
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9034 - loss: 0.3470 - val_accuracy: 0.9192 - val_loss: 0.2951
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9163 - loss: 0.2964 - val_accuracy: 0.9286 - val_loss: 0.2611
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9251 - loss: 0.2658 - val_accuracy: 0.9350 - val_loss: 0.2383
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9318 - loss: 0.2435 - val_accuracy: 0.9396 - val_loss: 0.2210
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9364 - loss: 0.2257 - val_accuracy: 0.9448 - val_loss: 0.2071
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9404 - loss: 0.2108 - val_accuracy: 0.9488 - val_loss: 0.1955
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9437 - loss: 0.1981 - val_accurac

INFO:tensorflow:Assets written to: my_mnist_model/0001/assets


Saved artifact at 'my_mnist_model/0001'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  130455536472272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130455536473808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130459838358992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130459838357264: TensorSpec(shape=(), dtype=tf.resource, name=None)


TensorFlow guarda:

- Arquitectura de la red.
- Pesos aprendidos.
- Configuración.
- Funciones de inferencia.
  

Todo junto.

¿Qué aparece en disco?

Algo parecido a:

my_mnist_model/

└── 0001/

    ├── saved_model.pb
    
    ├── assets/
    
    └── variables/
    
- saved_model.pb -> Contiene la descripción del grafo TensorFlow: estructura de la red, ops TF, firmas de entrada y salida, grafo de inferencia -> es un plano del modelo

- variables/ -> Contiene los pesos entrenados.

- assets/ -> Archivos auxiliares si existen.-> Vocabolario NLP, diccionarios, tabala de lookup (aquí no hay nada de eso

- fingerprint.pb ->Identificar de forma única el modelo, comprobar integridad, detectar cambios. Nuevo. Aparece en keras 3

Importante-> El modelo ya no es un objeto de Python sino un artefacto desplegable (como un .exe)

Let's take a look at the file tree (we've discussed what each of these file is used for in chapter 10):

In [7]:
# Una forma de ver qué archivos y carpetas ha creado TensorFlow al exportar el modelo.

sorted([str(path) for path in model_path.parent.glob("**/*")])  # extra code

# model_path.parent -> Lo que está encima del nivel donde se aloja el modelo -> my_mnist_model
# model_path.parent.glob("**/*") -> busca recursivamente todo lo que hay dentro de my_mnist_model.
# [str(path) for path in ...] -> convierte cada objeto Path en una cadena de texto.
# sorted() -> Ordena alfabéticamente

['my_mnist_model/0001',
 'my_mnist_model/0001/assets',
 'my_mnist_model/0001/fingerprint.pb',
 'my_mnist_model/0001/saved_model.pb',
 'my_mnist_model/0001/variables',
 'my_mnist_model/0001/variables/variables.data-00000-of-00001',
 'my_mnist_model/0001/variables/variables.index',
 'my_mnist_model/0002',
 'my_mnist_model/0002/assets',
 'my_mnist_model/0002/fingerprint.pb',
 'my_mnist_model/0002/saved_model.pb',
 'my_mnist_model/0002/variables',
 'my_mnist_model/0002/variables/variables.data-00000-of-00001',
 'my_mnist_model/0002/variables/variables.index']

Let's inspect the SavedModel:

In [8]:
# inspeccionar el SavedModel desde fuera de Python.

!saved_model_cli show --dir '{model_path}'

I0000 00:00:1782149432.708949    1855 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782149432.747681    1855 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782149433.736752    1855 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
The given SavedModel contains the following tag-sets:
'serve'


In [9]:
!saved_model_cli show --dir '{model_path}' --tag_set serve

I0000 00:00:1782149436.199217    1907 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782149436.233447    1907 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782149437.102278    1907 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
The given SavedModel MetaGraphDef contains SignatureDefs with the following keys:
SignatureDef key: "__saved_model_init_op"
SignatureD

In [10]:
!saved_model_cli show --dir '{model_path}' --tag_set serve \
                      --signature_def serving_default

I0000 00:00:1782149439.324717    1960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782149439.361098    1960 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782149440.251349    1960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
The given SavedModel SignatureDef contains the following input(s):
  inputs['keras_tensor'] tensor_info:
      dtype: DT_FLOAT
      s

For even more details, you can run the following command:

```ipython
!saved_model_cli show --dir '{model_path}' --all
```

### Installing and Starting TensorFlow Serving

If you are running this notebook in Colab or Kaggle, TensorFlow Server needs to be installed:

In [11]:
if "google.colab" in sys.modules or "kaggle_secrets" in sys.modules:
    url = "https://storage.googleapis.com/tensorflow-serving-apt"
    src = "stable tensorflow-model-server tensorflow-model-server-universal"
    !echo 'deb {url} {src}' > /etc/apt/sources.list.d/tensorflow-serving.list
    !curl '{url}/tensorflow-serving.release.pub.gpg' | apt-key add -
    !apt update -q && apt-get install -y tensorflow-model-server
    %pip install -q -U tensorflow-serving-api

If `tensorflow_model_server` is installed (e.g., if you are running this notebook in Colab), then the following 2 cells will start the server. If your OS is Windows, you may need to run the `tensorflow_model_server` command in a terminal, and replace `${MODEL_DIR}` with the full path to the `my_mnist_model` directory.

In [12]:
import os

os.environ["MODEL_DIR"] = str(model_path.parent.absolute())
# crea una variable de entorno llamada MODEL_DIR con la ruta absoluta donde está mi modelo.

Creamos un servidor  de inferencia independiente del proceso python del jupyter

In [13]:
%%bash --bg 
# Arranca TensorFlow Serving en segundo plano desde Jupyter.
tensorflow_model_server \
    --port=8500 \
    --rest_api_port=8501 \
    --model_name=my_mnist_model \
    --model_base_path="/mnt/d/Repos Github/handson-ml3/my_mnist_model" >my_server.log 2>&1 
# TF Serving buscará y cargará 0001 automáticamnte
# >my_server.log 2>&1 -> Redirige la salida estandar y el error estandar al archivo my_serve.log
# Antes tenía --model_base_path="${MODEL_DIR}" >my_server.log 2>&1 pero mi dirección tiene un espacio (Repos Github)
#TF Serving se está ejecutando en el fondo y sus registros se guardan en my_server.log
# Ha cargado nuestro modelo MNIST (version 1) y está esperando solicityudes gRPC y REST en los puertos 8500 y 8501

- TF Serving buscará y cargará 0001 automáticamnte
- >my_server.log 2>&1 -> Redirige la salida estandar y el error estandar al archivo my_serve.log
- Antes tenía --model_base_path="${MODEL_DIR}" >my_server.log 2>&1 pero mi dirección tiene un espacio (Repos Github)
- TF Serving se está ejecutando en el fondo y sus registros se guardan en my_server.log
- Ha cargado nuestro modelo MNIST (version 1) y está esperando solicityudes gRPC y REST en los puertos 8500 y 8501

In [14]:
import time

time.sleep(2) # let's wait a couple seconds for the server to start

Significa:

"Espera 2 segundos antes de continuar."

¿Por qué?

Porque acabas de arrancar TensorFlow Serving en segundo plano y el servidor necesita un pequeño tiempo para:

- Arrancar el proceso.
- Leer el SavedModel.
- Cargar los pesos.
- Abrir los puertos 8500 y 8501.

Si inmediatamente después haces una petición REST:
puede ocurrir que el servidor aún no haya terminado de arrancar y obtengas:

Connection refused

If you are running this notebook on your own machine, and you prefer to install TF Serving using Docker, first make sure [Docker](https://docs.docker.com/install/) is installed, then run the following commands in a terminal. You must replace `/path/to/my_mnist_model` with the appropriate absolute path to the `my_mnist_model` directory, but do not modify the container path `/models/my_mnist_model`.

```bash
docker pull tensorflow/serving  # downloads the latest TF Serving image

docker run -it --rm -v "/path/to/my_mnist_model:/models/my_mnist_model" \
    -p 8500:8500 -p 8501:8501 -e MODEL_NAME=my_mnist_model tensorflow/serving
```

### Querying TF Serving through the REST API

Next, let's send a REST query to TF Serving:

In [15]:
import json

X_new = X_test[:3]  # pretend we have 3 new digit images to classify shape (3,28,28)
request_json = json.dumps({
    "signature_name": "serving_default",
    "instances": X_new.tolist(),
})

# json.dumps() -> diccionario - json
# json.loads() -> json - diccionario
# X_new.tolist() -> convierte X_new de matriz de numpy a listas normales de Python
# Cada lista de lista (3 en total) tiene 28  listas y cada lista tiene 28 elementos

# Con request_jason estamos construyendo un mensaje para viajar por HTTP


In [16]:
X_new

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(3, 28, 28), dtype=uint8)

In [17]:
request_json

'{"signature_name": "serving_default", "instances": [[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 84, 185, 159, 151, 60, 36, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 222, 254, 254, 254, 254, 241, 198, 198, 198, 198, 198, 198, 198, 198, 170, 52, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 67, 114, 72, 114, 163, 227, 254, 225, 254, 254, 254, 250, 229, 254, 254, 140, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 

In [18]:
request_json[:100] + "..." + request_json[-10:] # concatena trozos de un mismo string

'{"signature_name": "serving_default", "instances": [[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..., 0, 0]]]}'

Now let's use TensorFlow Serving's REST API to make predictions:

En API REST:
- Get - > Obtener datos
- Post -> Crear o enviar datos
- Put-> Actualizar algo
- Delete -> Borrar algo

In [19]:
# Llamamos al moodelo como si fuera un servicio en Internet
# REST API -> puerto 8501

import requests

server_url = "http://127.0.0.1:8501/v1/models/my_mnist_model:predict"
# http:// -> protocolo HTTP
# localhost -> mi ordenador. He puesto la ip de esta máquina (127.0.0.1) porque me daba error
# 8501 -> Puerto REST API
# /v1/models/my_mnist_model -> Nombre del modelo
    #/v1 -> Versión de la API REST
    #/models -> Recurso "modelos" -> "voy a interactuar con un modelo"
# :predict -> Operación que quiero realizar

# Enviar la petición
response = requests.post(server_url, data=request_json) # .post -> enviar datos.
# Antes pred = model.predict(X_new); Ahora request.post()
response.raise_for_status()  # raise an exception in case of error. Comprobar errores
response = response.json() # convertir json  a diccionario. 
# La variable response NO es un string JSON sino un objeto de la librería Requests

# Cuando TnesorFlow Serving recibe el json, no llama directamnete al modelo. Primero conveirte a tensor las instancias del jason
# Reconstruyendo así (3, 3, 28)
# Sabe que tiene que hacer esa reconstrucción porque el SavedModel guarda la forma
    # inputs['keras_tensor']
    # shape: (-1, 28, 28)
    # dtype: DT_FLOAT

In [20]:
response.keys()

dict_keys(['predictions'])

In [21]:
import numpy as np

y_proba = np.array(response["predictions"])
# response["predictions"] es una lista de listas que pasamos a matriz numpy
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

- La red neuronal habla tensores.
- La red habla JSON.
- TensorFlow Serving traduce entre ambos.

### Querying TF Serving through the gRPC API

- REST:
diccionario → JSON → HTTP/1.1 -> TensorFlow Serving

- gRPC:
PredictRequest → TensorProto → gRPC ->HTTP/2 -> TensorFlow Serving Aquí no vemos URLs

In [22]:
# Contruimos el "paquete" que se va a cnviar por gRPC a TensorFlow Serving
from tensorflow_serving.apis.predict_pb2 import PredictRequest

# 1. Creamos una petición gRPC
request = PredictRequest() # este request dice "quiero hacer una predicción"

# 2. Decimos qué modelo queremos usar
request.model_spec.name = model_name #definido antes model_name = "my_mnist_model"

# 3. Decimos qué firma del modelo queremos llamar
request.model_spec.signature_name = "serving_default"

# 4. Metemos la entrada dentro de petición como TensorProto
input_name = "keras_tensor_6"  # == "flatten_input" Aparece en la firma de entrada de saved_model_cli
request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X_new.astype("float32")))
# aquí se produce traducción X_New: Tensor ->TensorProto

In [23]:
# Enviamos la petición gRPC

import grpc
from tensorflow_serving.apis import prediction_service_pb2_grpc

channel = grpc.insecure_channel("127.0.0.1:8500") # abre un canal hacia el servidor gRPC. El puerto 8500 es el de gRPC, no el de REST.
predict_service = prediction_service_pb2_grpc.PredictionServiceStub(channel) # crea el cliente Python que sabe hablar con TensorFlow Serving.
response = predict_service.Predict(request, timeout=10.0) # se envía la petición real

Convert the response to a tensor:

Hacemos ahora el camni inverso:

gRPC -> TensorProto ->Numpy

In [24]:
output_name = "output_0" # Lo pillo de !saved_model_cli show. Ver esa celda
outputs_proto = response.outputs[output_name] # extrae de la respuesta gRPC el TensorProto asociado a la salida.
y_proba = tf.make_ndarray(outputs_proto) # convierte en ndarray -> operaión inversa a tf.make_tensor_proto(X_new)

In [25]:
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]],
      dtype=float32)

If your client does not include the TensorFlow library, you can convert the response to a NumPy array like this:

In [26]:
# extra code – shows how to avoid using tf.make_ndarray()
output_name = "output_0"
outputs_proto = response.outputs[output_name]
shape = [dim.size for dim in outputs_proto.tensor_shape.dim]
y_proba = np.array(outputs_proto.float_val).reshape(shape)
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

### Deploying a new model version

In [27]:
# extra code – build and train a new MNIST model version
np.random.seed(42)
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8),
    tf.keras.layers.Rescaling(scale=1 / 255),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

Epoch 1/10


/home/srmjf/tf/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1782149444.736443     566 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_76841__.13
I0000 00:00:1782149445.402964     566 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1782149445.908481    3874 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_22', 164 bytes spill stores, 164 bytes spill loads



1707/1719 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6563 - loss: 1.1960

I0000 00:00:1782149450.969286     566 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_76841__.13


1719/1719 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8051 - loss: 0.7174 - val_accuracy: 0.9070 - val_loss: 0.3430
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9090 - loss: 0.3214 - val_accuracy: 0.9284 - val_loss: 0.2667
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9229 - loss: 0.2674 - val_accuracy: 0.9370 - val_loss: 0.2293
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9329 - loss: 0.2338 - val_accuracy: 0.9436 - val_loss: 0.2037
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9401 - loss: 0.2084 - val_accuracy: 0.9494 - val_loss: 0.1845
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9456 - loss: 0.1886 - val_accuracy: 0.9540 - val_loss: 0.1702
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9499 - loss: 0.1728 - val_accuracy: 0.9566 - val_loss: 0.1591
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9536 - loss: 0.1599 - val_accura

In [28]:
model_version = "0002"
model_path = Path(model_name) / model_version
model.export(model_path)

INFO:tensorflow:Assets written to: my_mnist_model/0002/assets


INFO:tensorflow:Assets written to: my_mnist_model/0002/assets


Saved artifact at 'my_mnist_model/0002'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='keras_tensor_6')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  130455093240528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130455093241296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130455093242064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130455093241488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130455093242448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130455093241872: TensorSpec(shape=(), dtype=tf.resource, name=None)


La idea importante es que el servidor apunta a: La idea importante es que el servidor apunta a:my_mnist_model, no a my_mnist_model/0001

Por eso puede cambiar de versión sin cambiar la URL del cliente

Let's take a look at the file tree again:

In [29]:
sorted([str(path) for path in model_path.parent.glob("**/*")])  # extra code

['my_mnist_model/0001',
 'my_mnist_model/0001/assets',
 'my_mnist_model/0001/fingerprint.pb',
 'my_mnist_model/0001/saved_model.pb',
 'my_mnist_model/0001/variables',
 'my_mnist_model/0001/variables/variables.data-00000-of-00001',
 'my_mnist_model/0001/variables/variables.index',
 'my_mnist_model/0002',
 'my_mnist_model/0002/assets',
 'my_mnist_model/0002/fingerprint.pb',
 'my_mnist_model/0002/saved_model.pb',
 'my_mnist_model/0002/variables',
 'my_mnist_model/0002/variables/variables.data-00000-of-00001',
 'my_mnist_model/0002/variables/variables.index']

**Warning**: You may need to wait a minute before the new model is loaded by TensorFlow Serving.

In [30]:
import requests

server_url = "http://127.0.0.1:8501/v1/models/my_mnist_model:predict"
            
response = requests.post(server_url, data=request_json)
response.raise_for_status()
response = response.json()

In [31]:
response.keys()

dict_keys(['predictions'])

In [32]:
y_proba = np.array(response["predictions"])
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

## Creating a Prediction Service on Vertex AI


# Flujo típico de despliegue con Vertex AI

```text
Tu Jupyter local
   ↓
Entrenas un modelo
   ↓
Lo guardas / exportas
   ↓
Lo subes a Google Cloud Storage
   ↓
Vertex AI registra ese modelo
   ↓
Vertex AI crea un endpoint
   ↓
Despliega el modelo en ese endpoint
   ↓
Envías datos y recibes predicciones
```

## Equivalencia conceptual

| Local | Vertex AI |
|---------|---------|
| Jupyter | Jupyter |
| SavedModel | SavedModel |
| Disco local | Cloud Storage |
| TensorFlow Serving | Vertex AI Endpoint |
| Docker | Infraestructura gestionada por Google |
| REST / gRPC | Endpoint de Vertex AI |
| Predicciones | Predicciones |

## Idea principal

Vertex AI no cambia el modelo.

Lo que hace es gestionar automáticamente:

- Almacenamiento del modelo.
- Versionado.
- Infraestructura.
- Escalado.
- Endpoints.
- Monitorización.
- Seguridad.

El flujo lógico sigue siendo:

```text
Datos
   ↓
Entrenamiento
   ↓
Modelo
   ↓
Despliegue
   ↓
Predicciones
```

Follow the instructions in the book to create a Google Cloud Platform account and activate the Vertex AI and Cloud Storage APIs. Then, if you're running this notebook in Colab, you can run the following cell to authenticate using the same Google account as you used with Google Cloud Platform, and authorize this Colab to access your data.

**WARNING: only do this if you trust this notebook!**
* Be extra careful if this is not the official notebook from https://github.com/ageron/handson-ml3: the Colab URL should start with https://colab.research.google.com/github/ageron/handson-ml3. Or else, the code could do whatever it wants with your data.

If you are not running this notebook in Colab, you must follow the instructions in the book to create a service account and generate a key for it, download it to this notebook's directory, and name it `my_service_account_key.json` (or make sure the `GOOGLE_APPLICATION_CREDENTIALS` environment variable points to your key).

In [39]:
pip install google-auth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 27.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 15.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [40]:
# Primero hay que autenticarse -> Pero para eso necesito tener instalada la CLI de Google Cloud.

from google.auth import default
credentials, project = default()


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [30]:
project_id = "my_project"  ##### CHANGE THIS TO YOUR PROJECT ID ##### el que tengamos en google

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif "kaggle_secrets" in sys.modules:
    from kaggle_secrets import UserSecretsClient
    UserSecretsClient().set_gcloud_credentials(project=project_id)
else:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my_service_account_key.json"

In [31]:
 #Aquí se crea un almacenamiento en la nube donde guardaré el modelo para que Vertex AI pueda acceder a él.

from google.cloud import storage

bucket_name = "my_bucket"  ##### CHANGE THIS TO A UNIQUE BUCKET NAME ##### Elijo el nombre aquí
# Este nombre tiene que ser único a nivel general
location = "us-central1"

storage_client = storage.Client(project=project_id)
bucket = storage_client.create_bucket(bucket_name, location=location)
#bucket = storage_client.bucket(bucket_name)  # to reuse a bucket instead

In [32]:
# Subimos el modelo local al bucket de Google Cloud Storage 

def upload_directory(bucket, dirpath): # -> Sube una carpeta completa al bucket.
    dirpath = Path(dirpath) # convierte nuestra carpeta en un objeto path
    for filepath in dirpath.glob("**/*"): # Recorre todos los archivos y subdirectorios.
        if filepath.is_file(): # ignora directorios, sólo procesa archivos
            blob = bucket.blob(filepath.relative_to(dirpath.parent).as_posix())
            blob.upload_from_filename(filepath) # Sube tantos blobs como archivos -> saved_model.pb, variables.data, variables.indez...

upload_directory(bucket, "my_mnist_model")

In [33]:
# extra code – a much faster multithreaded implementation of upload_directory()
#              which also accepts a prefix for the target path, and prints stuff

from concurrent import futures

def upload_file(bucket, filepath, blob_path):
    blob = bucket.blob(blob_path)
    blob.upload_from_filename(filepath)

def upload_directory(bucket, dirpath, prefix=None, max_workers=50):
    dirpath = Path(dirpath)
    prefix = prefix or dirpath.name
    with futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_filepath = {
            executor.submit(
                upload_file,
                bucket, filepath,
                f"{prefix}/{filepath.relative_to(dirpath).as_posix()}"
            ): filepath
            for filepath in sorted(dirpath.glob("**/*"))
            if filepath.is_file()
        }
        for future in futures.as_completed(future_to_filepath):
            filepath = future_to_filepath[future]
            try:
                result = future.result()
            except Exception as ex:
                print(f"Error uploading {filepath!s:60}: {ex}")  # f!s is str(f)
            else:
                print(f"Uploaded {filepath!s:60}", end="\r")

    print(f"Uploaded {dirpath!s:60}")

Alternatively, if you installed Google Cloud CLI (it's preinstalled on Colab), then you can use the following `gsutil` command:

In [34]:
# Esto es lo de antes pero más rápido y eficiente en Bash ( Linux/WSL)

#!gsutil -m cp -r my_mnist_model gs://{bucket_name}/

# gsutil -> CLI de Google Cloud Storage
# cp -> Copiar
# -r -> Recursivo -> Copia toda la carpeta
# -m -> Multihilo -> Cada hilo hace un achivo -> Mucho más ráoido

In [35]:
from google.cloud import aiplatform # Importa una librería Python que sabe hablar con Vertex AI.

server_image = "gcr.io/cloud-aiplatform/prediction/tf2-gpu.2-8:latest"
    # gcr.io -> Repositorio de imágenes de Google
    # cloud-aiplatform/prediction -> Imágenes preparadas para Vertex AI Prediction
    # tf2-gpu.2-8 -> Entorno preparado para TensorFlow 2.8 con soporte GPU
    # latest -> Última versión
# server_image es un contendor que contiene las características del entorno que google va a usar para ejecutar mi modelo

aiplatform.init(project=project_id, location=location) # decimos con qué va aproyecto id va a trabajar y en qué location
mnist_model = aiplatform.Model.upload(
    display_name="mnist",
    artifact_uri=f"gs://{bucket_name}/my_mnist_model/0001", # dónde está el modelo en el bucket
    serving_container_image_uri=server_image, # Usa el entorno preparado por Google para servir modelos TensorFlow con GPU
)

Creating Model
Create Model backing LRO: projects/522977795627/locations/us-central1/models/4798114811986575360/operations/53403898236370944
Model created. Resource name: projects/522977795627/locations/us-central1/models/4798114811986575360
To use this Model in another session:
model = aiplatform.Model('projects/522977795627/locations/us-central1/models/4798114811986575360')


**Warning**: this cell may take several minutes to run, as it waits for Vertex AI to provision the compute nodes:

In [36]:
endpoint = aiplatform.Endpoint.create(display_name="mnist-endpoint") # cremaos un endpoin t
#  enpoint -> Punto de acceso desde donde los clientes puede pedir predicciones

endpoint.deploy(
    mnist_model,
    min_replica_count=1,
    max_replica_count=5, # Google puede crecer hasta 5 instancias (copias del modelo) Una instancia puede atneder a muchos cli etes
    machine_type="n1-standard-4",
    accelerator_type="NVIDIA_TESLA_K80", # reserva máquina
    accelerator_count=1
)

Creating Endpoint
Create Endpoint backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/4135354010494304256
Endpoint created. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/522977795627/locations/us-central1/endpoints/5133373499481522176')
Deploying Model projects/522977795627/locations/us-central1/models/4798114811986575360 to Endpoint : projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Deploy Endpoint model backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/388359120522051584
Endpoint model deployed. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176


Hemos pasado de Modelo registrado a Modelo disponible para recibir peticiones

El Endpoint ID es el identificador único del endpoint

In [37]:
response = endpoint.predict(instances=X_new.tolist())
# endpoint.predict(...) es parecido a model.predict(...) pero el modelo ya no está en mi ordenador

In [38]:
import numpy as np

np.round(response.predictions, 2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

In [39]:
endpoint.undeploy_all()  # undeploy all models from the endpoint Quita todos los modelos desplegados
# apaga las máquinas, libera cpus gpus deja de servir predicciones
# Importante para seguir gastando
endpoint.delete() # elimina el endpoint

Undeploying Endpoint model: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Undeploy Endpoint model backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/3579722406467469312
Endpoint model undeployed. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Deleting Endpoint : projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Delete Endpoint  backing LRO: projects/522977795627/locations/us-central1/operations/4738836360561950720
Endpoint deleted. . Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176


## Running Batch Prediction Jobs on Vertex AI

Tengo muchos datos  
↓  
Los guardo en un archivo  
↓  
Los subo al bucket  
↓  
Vertex AI procesa el archivo entero  
↓  
Me deja los resultados en otro sitio

In [40]:
# Preparamos el archivo de entrada

batch_path = Path("my_mnist_batch") # crea una carpeta local
batch_path.mkdir(exist_ok=True)

with open(batch_path / "my_mnist_batch.jsonl", "w") as jsonl_file: # crea un archivo JSON Lines (JSON por líneas)
    for image in X_test[:100].tolist(): # toma las primeras 100 imágenes, em formato lista
        jsonl_file.write(json.dumps(image)) # Pasa de lista de listas a json -> cada imagen va en una fila
        jsonl_file.write("\n") # salto de págian para próxima línea

upload_directory(bucket, batch_path)

Uploaded my_mnist_batch                                              


In [38]:
# Para hacerlo de manera más eficiente a través de bash:

# !gsutil cp my_mnist_batch/my_mnist_batch.jsonl \
#          gs://{bucket_name}/my_mnist_batch/

In [41]:
batch_prediction_job = mnist_model.batch_predict(
    job_display_name="my_batch_prediction_job",
    machine_type="n1-standard-4",
    starting_replica_count=1,
    max_replica_count=5,
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=1,
    gcs_source=[f"gs://{bucket_name}/{batch_path.name}/my_mnist_batch.jsonl"],# ruta de entrada
    gcs_destination_prefix=f"gs://{bucket_name}/my_mnist_predictions/", # Guarda las predicciones en otro directorio del bucket 
    sync=True  # set to False if you don't want to wait for completion
)

# en un endpoint las réplicas permanecen vivas 
# # en btach las réplicas existen mientras dura el trabajo

Creating BatchPredictionJob
BatchPredictionJob created. Resource name: projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/4346926367237996544?project=522977795627
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_PENDING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/

In [42]:
batch_prediction_job.output_info  # extra code – shows the output directory

gcs_output_directory: "gs://my_bucket/my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z"

In [43]:
# equivalente a response = endpoint.predict(...) en batch

y_probas = [] # preparamos lista vacía
for blob in batch_prediction_job.iter_outputs(): # recorremos blobs generados
    print(blob.name)  # extra code para ver qué archivos han aparecido.
    if "prediction.results" in blob.name: # sólo los que tienen results -> predicciones
        for line in blob.download_as_text().splitlines():
            y_proba = json.loads(line)["prediction"] # json.loads(line) convbierto json en lista de python
            y_probas.append(y_proba)

my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.errors_stats-00000-of-00001
my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.results-00000-of-00002
my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.results-00001-of-00002


In [44]:
y_pred = np.argmax(y_probas, axis=1) # obtiene la posición con más probabilidad de las 10 salidas. El númeor predicho
accuracy = np.sum(y_pred == y_test[:100]) / 100 # suma todos los True / Total de la muestra =100

In [45]:
accuracy

0.98

In [46]:
mnist_model.delete() # elimiba el modelo registrado

Deleting Model : projects/522977795627/locations/us-central1/models/4798114811986575360
Delete Model  backing LRO: projects/522977795627/locations/us-central1/operations/598902403101622272
Model deleted. . Resource name: projects/522977795627/locations/us-central1/models/4798114811986575360


Let's delete all the directories we created on GCS (i.e., all the blobs with these prefixes):

In [47]:
for prefix in ["my_mnist_model/", "my_mnist_batch/", "my_mnist_predictions/"]:
    blobs = bucket.list_blobs(prefix=prefix)
    for blob in blobs:
        blob.delete()

#bucket.delete()  # uncomment and run if you want to delete the bucket itself
batch_prediction_job.delete()

Deleting BatchPredictionJob : projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544
Delete BatchPredictionJob  backing LRO: projects/522977795627/locations/us-central1/operations/6699028098374959104
BatchPredictionJob deleted. . Resource name: projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544


# Deploying a Model to a Mobile or Embedded Device

In [48]:
converter = tf.lite.TFLiteConverter.from_saved_model(str(model_path))
# Lee SavedModel y Prepara conversión a TFLite
tflite_model = converter.convert() # convierte el modelo a formato,tflite
with open("my_converted_savedmodel.tflite", "wb") as f: # wb write binary
    f.write(tflite_model) guarda esos bytes en dico

2022-04-10 09:03:52.237094: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2022-04-10 09:03:52.237108: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2022-04-10 09:03:52.237830: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: my_mnist_model/0001
2022-04-10 09:03:52.238869: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2022-04-10 09:03:52.238881: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: my_mnist_model/0001
2022-04-10 09:03:52.242108: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2022-04-10 09:03:52.263868: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: my_mnist_model/0001
2022-04-10 09:03:52.271298: I tensorflow/cc/saved_model/loader.cc:301] SavedModel load for tags { serve }; Status: success: OK. Too

In [49]:
# extra code – shows how to convert a Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

In [50]:
# Para cuantificación postentrenamientro
geron habla de la posibilidad de 
converter.optimizations = [tf.lite.Optimize.DEFAULT]

In [51]:
tflite_model = converter.convert()
with open("my_converted_keras_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs/assets


INFO:tensorflow:Assets written to: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs/assets
2022-04-10 09:26:30.319286: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2022-04-10 09:26:30.319301: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2022-04-10 09:26:30.319417: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs
2022-04-10 09:26:30.320420: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2022-04-10 09:26:30.320431: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs
2022-04-10 09:26:30.323773: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2022-04-10 09:26:30.345416: I tensorflow/cc/saved_model/loader.cc:212] Running initialization

# Running a Model in a Web Page

Code examples for this section are hosted on glitch.com, a website that lets you create Web apps for free.

* https://homl.info/tfjscode: a simple TFJS Web app that loads a pretrained model and classifies an image.
* https://homl.info/tfjswpa: the same Web app setup as a WPA. Try opening this link on various platforms, including mobile devices.
** https://homl.info/wpacode: this WPA's source code.
* https://tensorflow.org/js: The TFJS library.
** https://www.tensorflow.org/js/demos: some fun demos.

# Using GPUs to Speed Up Computations

In [35]:
!nvidia-smi # Para ver qué GPU usp

Fri Jun 19 09:42:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 596.49         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...    On  |   00000000:01:00.0  On |                  N/A |
| N/A   51C    P8              6W /   89W |   11317MiB /  12282MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Let's check that TensorFlow can see the GPU:

In [36]:
# Para comprobar TF ve mi GPU
physical_gpus = tf.config.list_physical_devices("GPU")
physical_gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

If you want your TensorFlow script to use only GPUs \#0 and \#1 (based on PCI order), then you can set the environment variables `CUDA_DEVICE_ORDER=PCI_BUS_ID` and `CUDA_VISIBLE_DEVICES=0,1` before starting your script, or in the script itself before using TensorFlow.

## Managing the GPU RAM

To limit the amount of RAM to 2GB per GPU:

In [53]:
#for gpu in physical_gpus:
#    tf.config.set_logical_device_configuration(
#        gpu,
#        [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
#    )

To make TensorFlow grab memory as it needs it (only releasing it when the process shuts down):

In [54]:
#for gpu in physical_gpus:
#    tf.config.experimental.set_memory_growth(gpu, True)

Equivalently, you can set the `TF_FORCE_GPU_ALLOW_GROWTH` environment variable to `true` before using TensorFlow.

To split a physical GPU into two logical GPUs:

In [55]:
#tf.config.set_logical_device_configuration(
#    physical_gpus[0],
#    [tf.config.LogicalDeviceConfiguration(memory_limit=2048),
#     tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
#)

In [56]:
logical_gpus = tf.config.list_logical_devices("GPU")
logical_gpus

[LogicalDevice(name='/device:GPU:0', device_type='GPU')]


## Placing Operations and Variables on Devices

To log every variable and operation placement (this must be run just after importing TensorFlow):

In [57]:
#tf.get_logger().setLevel("DEBUG")  # log level is INFO by default
#tf.debugging.set_log_device_placement(True)

In [37]:
a = tf.Variable([1., 2., 3.])  # float32 variable goes to the GPU
a.device

'/job:localhost/replica:0/task:0/device:GPU:0'

In [59]:
b = tf.Variable([1, 2, 3])  # int32 variable goes to the CPU
b.device

'/job:localhost/replica:0/task:0/device:CPU:0'

You can place variables and operations manually on the desired device using a `tf.device()` context:

In [60]:
with tf.device("/cpu:0"):
    c = tf.Variable([1., 2., 3.])

c.device

'/job:localhost/replica:0/task:0/device:CPU:0'

If you specify a device that does not exist, or for which there is no kernel, TensorFlow will silently fallback to the default placement:

In [61]:
# extra code

with tf.device("/gpu:1234"):
    d = tf.Variable([1., 2., 3.])

d.device

"'/job:localhost/replica:0/task:0/device:GPU:0'"

If you want TensorFlow to throw an exception when you try to use a device that does not exist, instead of falling back to the default device:

In [62]:
tf.config.set_soft_device_placement(False)

# extra code
try:
    with tf.device("/gpu:1000"):
        d = tf.Variable([1., 2., 3.])
except tf.errors.InvalidArgumentError as ex:
    print(ex)

tf.config.set_soft_device_placement(True)  # extra code – back to soft placement

Could not satisfy device specification '/job:localhost/replica:0/task:0/device:GPU:1000'. enable_soft_placement=0. Supported device types [CPU]. All available devices [/job:localhost/replica:0/task:0/device:CPU:0].


## Parallel Execution Across Multiple Devices

If you want to set the number of inter-op or intra-op threads (this may be useful if you want to avoid saturating the CPU, or if you want to make TensorFlow single-threaded, to run a perfectly reproducible test case):

In [63]:
#tf.config.threading.set_inter_op_parallelism_threads(10)
#tf.config.threading.set_intra_op_parallelism_threads(10)

# Training Models Across Multiple Devices

## Training at Scale Using the Distribution Strategies API

In [33]:
# extra code – creates a CNN model for MNIST using Keras
def create_model():
    return tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])

Modelo normal de Keras  
↓  
MirroredStrategy  
↓  
Una copia del modelo en cada GPU  
↓  
Cada GPU procesa una parte del batch  
↓  
Se combinan gradientes  
↓  
Una actualización común

In [34]:
tf.random.set_seed(42)

strategy = tf.distribute.MirroredStrategy() # crea una estrategia de distribución. “Mirrored” significa: Modelo espejo en cada GPU

with strategy.scope():

    #Todo lo que crees dentro queda bajo esa estrategia. Por eso van dentro:
    
    model = create_model()  # create a Keras model normally -> función anteior que crea el modelo
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])  # compile the model normally
    # TensorFlow necesita crear las variables del modelo sabiendo que van a estar replicadas.

batch_size = 100  # preferably divisible by the number of replicas
model.fit(X_train, y_train, epochs=10,
          validation_data=(X_valid, y_valid), batch_size=batch_size)
# .fit dividirá automáticamnete cada lote de ntrenamiento entre las réplicas (número de GPUs disponibles

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
/home/srmjf/tf/lib/python3.12/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


I0000 00:00:1781888515.513852     446 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


...

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


550/550 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.5991 - loss: 1.2506 - val_accuracy: 0.9014 - val_loss: 0.3593
Epoch 2/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8609 - loss: 0.4544 - val_accuracy: 0.9448 - val_loss: 0.1940
Epoch 3/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9112 - loss: 0.3019 - val_accuracy: 0.9620 - val_loss: 0.1420
Epoch 4/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9307 - loss: 0.2391 - val_accuracy: 0.9688 - val_loss: 0.1119
Epoch 5/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9417 - loss: 0.2017 - val_accuracy: 0.9724 - val_loss: 0.0963
Epoch 6/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9500 - loss: 0.1741 - val_accuracy: 0.9760 - val_loss: 0.0811
Epoch 7/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9544 - loss: 0.1587 - val_accuracy: 0.9784 - val_loss: 0.0796
Epoch 8/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9574 - loss: 0.1465 - val_accuracy: 0.9796 - val_

In [35]:
type(model.weights[0]) # ¿De qué tipo es el primer peso del modelo?
# model.weights -> lista con todos los pesos entrenables y no entrenables.
    # weights[0] → matriz de pesos capa 1
    # weights[1] → bias capa 1
    # weights[2] → matriz de pesos capa 2
    # weights[3] → bias capa 2


keras.src.backend.Variable

In [36]:
model.predict(X_new).round(2)  # extra code – the batch is split across all replicas

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step


array([[0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [42]:
# extra code – shows that saving a model does not preserve its distribution
#              strategy
model.save("my_mirrored_model.keras")
model = tf.keras.models.load_model("my_mirrored_model.keras")
type(model.weights[0])

keras.src.backend.Variable

In [44]:
with strategy.scope():
    model = tf.keras.models.load_model("my_mirrored_model.keras")

In [45]:
type(model.weights[0])

keras.src.backend.Variable

If you want to specify the list of GPUs to use:

In [46]:
strategy = tf.distribute.MirroredStrategy(devices=["/gpu:0", "/gpu:1"])

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


If you want to change the default all-reduce algorithm:

In [47]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [72]:
strategy = tf.distribute.MirroredStrategy(
    cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


If you want to use the `CentralStorageStrategy`:

In [73]:
strategy = tf.distribute.experimental.CentralStorageStrategy()

INFO:tensorflow:ParameterServerStrategy (CentralStorageStrategy if you are using a single machine) with compute_devices = ['/job:localhost/replica:0/task:0/device:CPU:0'], variable_device = '/job:localhost/replica:0/task:0/device:CPU:0'


In [74]:
# To train on a TPU in Google Colab:
#if "google.colab" in sys.modules and "COLAB_TPU_ADDR" in os.environ:
#  tpu_address = "grpc://" + os.environ["COLAB_TPU_ADDR"]
#else:
#  tpu_address = ""
#resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu_address)
#tf.config.experimental_connect_to_cluster(resolver)
#tf.tpu.experimental.initialize_tpu_system(resolver)
#strategy = tf.distribute.experimental.TPUStrategy(resolver)

## Training a Model on a TensorFlow Cluster

A TensorFlow cluster is a group of TensorFlow processes running in parallel, usually on different machines, and talking to each other to complete some work, for example training or executing a neural network. Each TF process in the cluster is called a "task" (or a "TF server"). It has an IP address, a port, and a type (also called its role or its job). The type can be `"worker"`, `"chief"`, `"ps"` (parameter server) or `"evaluator"`:
* Each **worker** performs computations, usually on a machine with one or more GPUs.
* The **chief** performs computations as well, but it also handles extra work such as writing TensorBoard logs or saving checkpoints. There is a single chief in a cluster. If it is not defined, then it is worker #0.
* A **parameter server** (ps) only keeps track of variable values, it is usually on a CPU-only machine.
* The **evaluator** obviously takes care of evaluation. There is usually a single evaluator in a cluster.

The set of tasks that share the same type is often called a "job". For example, the "worker" job is the set of all workers.

To start a TensorFlow cluster, you must first define it. This means specifying all the tasks (IP address, TCP port, and type). For example, the following cluster specification defines a cluster with 3 tasks (2 workers and 1 parameter server). It's a dictionary with one key per job, and the values are lists of task addresses:

In [75]:
# Arquitectura clásica de TensorFlow distribuido con workers y parameter servers (ps)

cluster_spec = {
    "worker": [
        "machine-a.example.com:2222",     # /job:worker/task:0
        "machine-b.example.com:2222"      # /job:worker/task:1
    ],
    "ps": ["machine-a.example.com:2221"]  # /job:ps/task:0
}

Every task in the cluster may communicate with every other task in the server, so make sure to configure your firewall to authorize all communications between these machines on these ports (it's usually simpler if you use the same port on every machine).

When a task is started, it needs to be told which one it is: its type and index (the task index is also called the task id). A common way to specify everything at once (both the cluster spec and the current task's type and id) is to set the `TF_CONFIG` environment variable before starting the program. It must be a JSON-encoded dictionary containing a cluster specification (under the `"cluster"` key), and the type and index of the task to start (under the `"task"` key). For example, the following `TF_CONFIG` environment variable defines the same cluster as above, with 2 workers and 1 parameter server, and specifies that the task to start is worker \#0:

In [76]:
os.environ["TF_CONFIG"] = json.dumps({
    "cluster": cluster_spec,
    "task": {"type": "worker", "index": 0}
})

'''
"Hola.
Estás dentro de este cluster.
Y tú eres el worker 0., el que inicia la tarea"
'''

Some platforms (e.g., Google Vertex AI) automatically set this environment variable for you.

TensorFlow's `TFConfigClusterResolver` class reads the cluster configuration from this environment variable:

In [77]:
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
resolver.cluster_spec()

ClusterSpec({'ps': ['machine-a.example.com:2221'], 'worker': ['machine-a.example.com:2222', 'machine-b.example.com:2222']})

In [78]:
resolver.task_type

'worker'

In [79]:
resolver.task_id

0

Now let's run a simpler cluster with just two worker tasks, both running on the local machine. We will use the `MultiWorkerMirroredStrategy` to train a model across these two tasks.

The first step is to write the training code. As this code will be used to run both workers, each in its own process, we write this code to a separate Python file, `my_mnist_multiworker_task.py`. The code is relatively straightforward, but there are a couple important things to note:
* We create the `MultiWorkerMirroredStrategy` before doing anything else with TensorFlow.
* Only one of the workers will take care of logging to TensorBoard. As mentioned earlier, this worker is called the *chief*. When it is not defined explicitly, then by convention it is worker #0.

La idea general del script es:

Cada worker ejecuta este mismo archivo  
↓  
Cada worker lee TF_CONFIG  
↓  
Cada worker sabe quién es  
↓  
Todos entrenan juntos  
↓  
Sólo el worker principal guarda el modelo definitivo

In [80]:
%%writefile my_mnist_multiworker_task.py # Guarda todo lo que viene debajo en un archivo llamado my_mnist_multiworker_task.py

import tempfile
import tensorflow as tf

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # at the start!

# Voy a entrenar con varios workers y todos mantendrán copias sincronizadas del modelo

resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver() # lee la variable: TF_CONFIG y averigua: Soy worker 0 o soy worker 1 o soy ps 0

print(f"Starting task {resolver.task_type} #{resolver.task_id}") # imprime salidas del resolver

# extra code – Load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

with strategy.scope(): # porque TensorFlow necesita crear las variables sabiendo que estarán distribuidas.
    model = tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10)

if resolver.task_id == 0:  # the chief saves the model to the right location Sólo seguarda lo que haga el jefe (id =0)
    model.save("my_mnist_multiworker_model", save_format="tf")
else:
    tmpdir = tempfile.mkdtemp()  # other workers save to a temporary directory. Gauradan tb para mantenber la sincronización
    model.save(tmpdir, save_format="tf")
    tf.io.gfile.rmtree(tmpdir)  # and we can delete this directory at the end!

import tempfile
import tensorflow as tf

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # at the start!
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
print(f"Starting task {resolver.task_type} #{resolver.task_id}")

# extra code – Load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10)

if resolver.task_id == 0:  # the chief saves the model to the right location
    model.export("my_mnist_multiworker_model")
# Porque en entrenamiento multiworker todos los workers ejecutan el mismo código.
# Si no pusieras el if, todos intentarían guardar el modelo en el mismo sitio.
else:
    tmpdir = tempfile.mkdtemp()  # other workers save to a temporary directory
    model.export("my_mnist_multiworker_model")
    tf.io.gfile.rmtree(tmpdir)  # and we can delete this directory at the end!
    # Borra un directorio entero y todo su contenido

Writing my_mnist_multiworker_task.py


worker 0  
↓  
guarda el modelo real  

otros workers  
↓  
guardan temporalmente  
↓  
borran esa carpeta

¿Por qué los otros workers tienen que guardar aunque sea temporalmente?

Porque en algunas estrategias de TensorFlow, todos los workers deben participar en la operación de guardado para mantener la sincronización. Pero sólo el principal, llamado normalmente chief, conserva el resultado final.

In a real world application, there would typically be a single worker per machine, but in this example we're running both workers on the same machine, so they will both try to use all the available GPU RAM (if this machine has a GPU), and this will likely lead to an Out-Of-Memory (OOM) error. To avoid this, we could use the `CUDA_VISIBLE_DEVICES` environment variable to assign a different GPU to each worker. Alternatively, we can simply disable GPU support, by setting `CUDA_VISIBLE_DEVICES` to an empty string.

We are now ready to start both workers, each in its own process. Notice that we change the task index:

In [81]:
%%bash --bg

export CUDA_VISIBLE_DEVICES=''
export TF_CONFIG='{"cluster": {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]},
                   "task": {"type": "worker", "index": 0}}'
python my_mnist_multiworker_task.py > my_worker_0.log 2>&1

In [82]:
%%bash --bg

export CUDA_VISIBLE_DEVICES=''
export TF_CONFIG='{"cluster": {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]},
                   "task": {"type": "worker", "index": 1}}'
python my_mnist_multiworker_task.py > my_worker_1.log 2>&1

**Note**: if you get warnings about `AutoShardPolicy`, you can safely ignore them. See [TF issue #42146](https://github.com/tensorflow/tensorflow/issues/42146) for more details.

That's it! Our TensorFlow cluster is now running, but we can't see it in this notebook because it's running in separate processes (but you can see the progress in `my_worker_*.log`).

Since the chief (worker #0) is writing to TensorBoard, we use TensorBoard to view the training progress. Run the following cell, then click on the settings button (i.e., the gear icon) in the TensorBoard interface and check the "Reload data" box to make TensorBoard automatically refresh every 30s. Once the first epoch of training is finished (which may take a few minutes), and once TensorBoard refreshes, the SCALARS tab will appear. Click on this tab to view the progress of the model's training and validation accuracy.

In [83]:
%load_ext tensorboard
%tensorboard --logdir=./my_mnist_multiworker_logs --port=6006

In [84]:
# strategy = tf.distribute.MultiWorkerMirroredStrategy(
#     communication_options=tf.distribute.experimental.CommunicationOptions(
#         implementation=tf.distribute.experimental.CollectiveCommunication.NCCL))
# Aquí decimos quecómo queremos que se comunique -> A través de NCCL

# En strategy = tf.distribute.MultiWorkerMirroredStrategy() -> TF elige automáticamente el método de comunicación


## Running Large Training Jobs on Vertex AI

Let's copy the training script, but add `import os` and change the save path to be the GCS path that the `AIP_MODEL_DIR` environment variable will point to:

In [85]:
%%writefile my_vertex_ai_training_task.py

import os
from pathlib import Path
import tempfile
import tensorflow as tf

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # at the start!
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()

if resolver.task_type == "chief":
    model_dir = os.getenv("AIP_MODEL_DIR")  # paths provided by Vertex AI. Lee las variables del entorno que vertex ha creado para mi
    tensorboard_log_dir = os.getenv("AIP_TENSORBOARD_LOG_DIR")
    checkpoint_dir = os.getenv("AIP_CHECKPOINT_DIR")
else:
    tmp_dir = Path(tempfile.mkdtemp())  # other workers use a temporary dirs demás workers crean una carpeta temporal
    model_dir = tmp_dir / "model"
    tensorboard_log_dir = tmp_dir / "logs"
    checkpoint_dir = tmp_dir / "ckpt"

callbacks = [tf.keras.callbacks.TensorBoard(tensorboard_log_dir),
             tf.keras.callbacks.ModelCheckpoint(checkpoint_dir)]

# Tensorboard guarda loss, accuracy, gráficas, historial
# Checkpoints guarda periódicamente pesos actruales
# extra code – Load and prepare the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# extra code – build and compile the Keras model using the distribution strategy
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Lambda(lambda X: X / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"), 
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10,
          callbacks=callbacks)
model.export(model_dir)

Writing my_vertex_ai_training_task.py


In [86]:
# Preparar un trabajo de entrenamiento en Vertex AI

custom_training_job = aiplatform.CustomTrainingJob(
    display_name="my_custom_training_job", # nombre que veré en la consola
    script_path="my_vertex_ai_training_task.py", # viene de celda anterior (ver primera línea)
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest", # ¿Qué entorno debe usar?
    model_serving_container_image_uri=server_image, # cuando termine el entrenamiento, ¿Con qué imagen haré inferencias
    # en celdas anteriores -> server_image = "gcr.io/cloud-aiplatform/prediction/tf2-gpu.2-8:latest"
    requirements=["gcsfs==2022.3.0"],  # not needed, this is just an example Lo mismon  que pip install gcsfs==2022.3.0
    #Google Cloud Storage File System -> Es una librería que permite tratar un bucket de Google Cloud Syorage como si fuera un sistema de archivos normal
    staging_bucket=f"gs://{bucket_name}/staging" #Este bucket se usa como zona temporal.
)

Cuando hagamos custom_training_job.run(...) - Vertex hará

1. Crear VM(s)

2. Descargar imagen TensorFlow

3. Instalar requirements

4. Subir script

5. Configurar TF_CONFIG

6. Lanzar workers

7. Entrenar

8. Guardar modelo

9. Apagar máquinas



In [87]:
# Crea las máquinas y empieza a trabajar
mnist_model2 = custom_training_job.run( 
    machine_type="n1-standard-4", ~máquinas de 4 CPUS de 3.75 G B RAM = 15GB RTAM
    replica_count=2, # worker 0 y worker 1. 2 máquinas trabajando juntas
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=2,
)

Training script copied to:
gs://my_bucket/aiplatform-2022-04-14-10:08:24.124-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://my_bucket/aiplatform-custom-training-2022-04-14-10:08:25.226 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5407999068506947584?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_PENDING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_RUNNING
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/6685701948726837248?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-c

Let's clean up:

In [88]:
mnist_model2.delete()
custom_training_job.delete()
blobs = bucket.list_blobs(prefix=f"gs://{bucket_name}/staging/")
for blob in blobs:
    blob.delete()

# Hyperparameter Tuning on Vertex AI

Este script es para que Vertex AI ejecute muchos entrenamientos parecidos, cambiando automáticamente algunos hiperparámetros, y luego compare cuál funciona mejor.

La idea general:

Trial 1 → n_hidden=1, n_neurons=128, lr=0.01, optimizer=adam
Trial 2 → n_hidden=3, n_neurons=256, lr=0.001, optimizer=sgd
Trial 3 → ...

Cada trial entrena un modelo distinto.

In [89]:
%%writefile my_vertex_ai_trial.py

import argparse

# argparse -> Biblioteca estándar de Python sirve para que un programa reciba parámetros de la línea de comda

parser = argparse.ArgumentParser() # establecemos los valores por defecti
parser.add_argument("--n_hidden", type=int, default=2)
parser.add_argument("--n_neurons", type=int, default=256)
parser.add_argument("--learning_rate", type=float, default=1e-2)
parser.add_argument("--optimizer", default="adam")
args = parser.parse_args()

import tensorflow as tf

def build_model(args):
    with tf.distribute.MirroredStrategy().scope():
        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8))
        for _ in range(args.n_hidden):
            model.add(tf.keras.layers.Dense(args.n_neurons, activation="relu"))
        model.add(tf.keras.layers.Dense(10, activation="softmax"))
        opt = tf.keras.optimizers.get(args.optimizer)
        opt.learning_rate = args.learning_rate
        model.compile(loss="sparse_categorical_crossentropy", optimizer=opt,
                      metrics=["accuracy"])
        return model

# extra code – loads and splits the dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# extra code – use the AIP_* environment variable and create the callbacks
import os
model_dir = os.getenv("AIP_MODEL_DIR")
tensorboard_log_dir = os.getenv("AIP_TENSORBOARD_LOG_DIR")
checkpoint_dir = os.getenv("AIP_CHECKPOINT_DIR")
trial_id = os.getenv("CLOUD_ML_TRIAL_ID")
tensorboard_cb = tf.keras.callbacks.TensorBoard(tensorboard_log_dir)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)
callbacks = [tensorboard_cb, early_stopping_cb]

model = build_model(args)
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                    epochs=10, callbacks=callbacks)
model.export(model_dir)  # extra code

import hypertune

hypertune = hypertune.HyperTune() # objeto para comunicarse con vertex
hypertune.report_hyperparameter_tuning_metric( # vertex apunta esta puntuación!
    hyperparameter_metric_tag="accuracy",  # name of the reported metric
    metric_value=max(history.history["val_accuracy"]),  # max accuracy value
    global_step=model.optimizer.iterations.numpy(),
)

Writing my_vertex_ai_trial.py


In [90]:
trial_job = aiplatform.CustomJob.from_local_script(  # Este es el trabajo base que voy a repetir muchas veces"
    display_name="my_search_trial_job",
    script_path="my_vertex_ai_trial.py",  # path to your training script
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",
    staging_bucket=f"gs://{bucket_name}/staging",
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=2,  # in this example, each trial will have 2 GPUs
)

Training script copied to:
gs://homl3-mybucket5/staging/aiplatform-2022-04-18-18:14:02.860-aiplatform_custom_trainer_script-0.1.tar.gz.


In [91]:
from google.cloud.aiplatform import hyperparameter_tuning as hpt

hp_job = aiplatform.HyperparameterTuningJob(
    display_name="my_hp_search_job",
    custom_job=trial_job,
    metric_spec={"accuracy": "maximize"}, # la máxima accuracy
    parameter_spec={ # minimos y maximos de nuestras posibles configuraciones
        "learning_rate": hpt.DoubleParameterSpec(min=1e-3, max=10, scale="log"),
        "n_neurons": hpt.IntegerParameterSpec(min=1, max=300, scale="linear"),
        "n_hidden": hpt.IntegerParameterSpec(min=1, max=10, scale="linear"),
        "optimizer": hpt.CategoricalParameterSpec(["sgd", "adam"]),
    },
    max_trial_count=100, # Como máximo 10 entrenamientos distintos
    parallel_trial_count=20, # hasta 20n entrenamientos simultáneos
)
hp_job.run()

Creating HyperparameterTuningJob
HyperparameterTuningJob created. Resource name: projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568
To use this HyperparameterTuningJob in another session:
hpt_job = aiplatform.HyperparameterTuningJob.get('projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568')
View HyperparameterTuningJob:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5825136187899117568?project=522977795627
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJ

La idea clave que debes recordar es:

my_vertex_ai_trial.py  
↓  
Cómo entrenar un modelo

trial_job  
↓  
Cómo ejecutar un trial

HyperparameterTuningJob  
↓  
Qué hiperparámetros explorar  
y cuántos trials lanzar

Esa última capa (`HyperparameterTuningJob`) es la que convierte un entrenamiento normal en una búsqueda automática de hiperparámetros.

In [92]:
def get_final_metric(trial, metric_id):
    for metric in trial.final_measurement.metrics:
        if metric.metric_id == metric_id:
            return metric.value

trials = hp_job.trials # devuelve una lista de obejtos Trial
trial_accuracies = [get_final_metric(trial, "accuracy") for trial in trials]
best_trial = trials[np.argmax(trial_accuracies)]

In [93]:
max(trial_accuracies)

0.977400004863739

In [94]:
best_trial.id

'98'

In [95]:
best_trial.parameters

[parameter_id: "learning_rate"
value {
  number_value: 0.001
}
, parameter_id: "n_hidden"
value {
  number_value: 8.0
}
, parameter_id: "n_neurons"
value {
  number_value: 216.0
}
, parameter_id: "optimizer"
value {
  string_value: "adam"
}
]

# Extra Material – Distributed Keras Tuner on Vertex AI

Instead of using Vertex AI's hyperparameter tuning service, you can use [Keras Tuner](https://keras.io/keras_tuner/) (introduced in Chapter 10) and run it on Vertex AI VMs. Keras Tuner provides a simple way to scale hyperparameter search by distributing it across multiple machines: it only requires setting three environment variables on each machine, then running your regular Keras Tuner code on each machine. You can use the exact same script on all machines. One of the machines acts as the chief, and the others act as workers. Each worker asks the chief which hyperparameter values to try—it acts as the oracle—then the worker trains the model using these hyperparameter values, and finally it reports the model's performance back to the chief, which can then decide which hyperparameter values the worker should try next.

The three environment variables you need to set on each machine are:

* `KERASTUNER_TUNER_ID`: equal to `"chief"` on the chief machine, or a unique identifier on each worker machine, such as `"worker0"`, `"worker1"`, etc.
* `KERASTUNER_ORACLE_IP`: the IP address or hostname of the chief machine. The chief itself should generally use `"0.0.0.0"` to listen on every IP address on the machine.
* `KERASTUNER_ORACLE_PORT`: the TCP port that the chief will be listening on.

You can use distributed Keras Tuner on any set of machines. If you want to run it on Vertex AI machines, then you can spawn a regular training job, and just modify the training script to set the three environment variables properly before using Keras Tuner.

For example, the script below starts by parsing the `TF_CONFIG` environment variable, which will be automatically set by Vertex AI, just like earlier. It finds the address of the task of type `"chief"`, and it extracts the IP address or hostname, and the TCP port. It then defines the tuner ID as the task type followed by the task index, for example `"worker0"`. If the tuner ID is `"chief0"`, it changes it to `"chief"`, and it sets the IP to `"0.0.0.0"`: this will make it listen on all IPv4 address on its machine. Then it defines the environment variables for Keras Tuner. Next, the script creates a tuner, just like in Chapter 10, the it runs the search, and finally it saves the best model to the location given by Vertex AI:

Antes: Vertex AI decide los trials
Ahora: Keras Tuner decide los trials

In [96]:
%%writefile my_keras_tuner_search.py

import json
import os

tf_config = json.loads(os.environ["TF_CONFIG"])

chief_ip, chief_port = tf_config["cluster"]["chief"][0].rsplit(":", 1)
tuner_id = f'{tf_config["task"]["type"]}{tf_config["task"]["index"]}'
if tuner_id == "chief0":
    tuner_id = "chief"
    chief_ip = "0.0.0.0"
    # extra code – since the chief doesn't work much, you can optimize compute
    # resources by running a worker on the same machine. To do this, you can
    # just make the chief start another process, after tweaking the TF_CONFIG
    # environment variable to set the task type to "worker" and the task index
    # to a unique value. Uncomment the next few lines to give this a try:
    # import subprocess
    # import sys
    # tf_config["task"]["type"] = "workerX"  # the worker on the chief's machine
    # os.environ["TF_CONFIG"] = json.dumps(tf_config)
    # subprocess.Popen([sys.executable] + sys.argv,
    #                  stdout=sys.stdout, stderr=sys.stderr)

os.environ["KERASTUNER_TUNER_ID"] = tuner_id # Quién soy
os.environ["KERASTUNER_ORACLE_IP"] = chief_ip # dónde está el jefe
os.environ["KERASTUNER_ORACLE_PORT"] = chief_port # en qué puerto escucha el ejefe

from pathlib import Path
import keras_tuner as kt
import tensorflow as tf

gcs_path = "/gcs/my_bucket/my_hp_search"  # replace with your bucket's name

def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2,
                             sampling="log")
    optimizer = hp.Choice("optimizer", values=["sgd", "adam"])
    if optimizer == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8))
    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])
    return model

hyperband_tuner = kt.Hyperband(
    build_model, objective="val_accuracy", seed=42,
    max_epochs=10, factor=3, hyperband_iterations=2,
    distribution_strategy=tf.distribute.MirroredStrategy(),
    directory=gcs_path, project_name="mnist")

# extra code – Load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

tensorboard_log_dir = os.environ["AIP_TENSORBOARD_LOG_DIR"] + "/" + tuner_id
tensorboard_cb = tf.keras.callbacks.TensorBoard(tensorboard_log_dir)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)
hyperband_tuner.search(X_train, y_train, epochs=10,
                       validation_data=(X_valid, y_valid),
                       callbacks=[tensorboard_cb, early_stopping_cb])

if tuner_id == "chief":
    best_hp = hyperband_tuner.get_best_hyperparameters()[0]
    best_model = hyperband_tuner.hypermodel.build(best_hp)
    best_model.save(os.getenv("AIP_MODEL_DIR"), save_format="tf")

Writing my_keras_tuner_search.py


Note that Vertex AI automatically mounts the `/gcs` directory to GCS, using the open source [GCS Fuse adapter](https://cloud.google.com/storage/docs/gcs-fuse). This gives us a shared directory across the workers and the chief, which is required by Keras Tuner. Also note that we set the distribution strategy to a `MirroredStrategy`. This will allow each worker to use all the GPUs on its machine, if there's more than one.


Replace `/gcs/my_bucket/` with <code>/gcs/<i>{bucket_name}</i>/</code>:

In [97]:
with open("my_keras_tuner_search.py") as f:
    script = f.read()

with open("my_keras_tuner_search.py", "w") as f:
    f.write(script.replace("/gcs/my_bucket/", f"/gcs/{bucket_name}/"))

Now all we need to do is to start a custom training job based on this script, exactly like in the previous section. Don't forget to add `keras-tuner` to the list of `requirements`:

In [98]:
hp_search_job = aiplatform.CustomTrainingJob(
    display_name="my_hp_search_job",
    script_path="my_keras_tuner_search.py",
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",
    model_serving_container_image_uri=server_image,
    requirements=["keras-tuner~=1.1.2"],
    staging_bucket=f"gs://{bucket_name}/staging",
)

In [99]:
mnist_model3 = hp_search_job.run(
    machine_type="n1-standard-4",
    replica_count=3,
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=2,
)

Training script copied to:
gs://my_bucket/staging/aiplatform-2022-04-15-13:34:32.591-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://my_bucket/staging/aiplatform-custom-training-2022-04-15-13:34:34.453 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/8601543785521872896?project=522977795627
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5022607048831926272?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/52297779562

And we have a model!

Let's clean up:

In [100]:
mnist_model3.delete()
hp_search_job.delete()
blobs = bucket.list_blobs(prefix=f"gs://{bucket_name}/staging/")
for blob in blobs:
    blob.delete()

# Extra Material – Using AutoML to Train a Model

Let's start by exporting the MNIST dataset to PNG images, and prepare an `import.csv` pointing to each image, and indicating the split (training, validation, or test) and the label:

In [101]:
import matplotlib.pyplot as plt

mnist_path = Path("datasets/mnist")
mnist_path.mkdir(parents=True, exist_ok=True)
idx = 0
with open(mnist_path / "import.csv", "w") as import_csv:
    for split, X, y in zip(("training", "validation", "test"),
                           (X_train, X_valid, X_test),
                           (y_train, y_valid, y_test)):
        for image, label in zip(X, y):
            print(f"\r{idx + 1}/70000", end="")
            filename = f"{idx:05d}.png"
            plt.imsave(mnist_path / filename, np.tile(image, 3))
            line = f"{split},gs://{bucket_name}/mnist/{filename},{label}\n"
            import_csv.write(line)
            idx += 1

70000/70000

Let's upload this dataset to GCS:

In [102]:
upload_directory(bucket, mnist_path)

Uploaded datasets/mnist                                              


Now let's create a managed image dataset on Vertex AI:

In [103]:
from aiplatform.schema.dataset.ioformat.image import single_label_classification

mnist_dataset = aiplatform.ImageDataset.create(
    display_name="mnist-dataset",
    gcs_source=[f"gs://{bucket_name}/mnist/import.csv"],
    project=project_id,
    import_schema_uri=single_label_classification,
    sync=True,
)

Creating ImageDataset
Create ImageDataset backing LRO: projects/522977795627/locations/us-central1/datasets/7532459492777132032/operations/3812233931370004480
ImageDataset created. Resource name: projects/522977795627/locations/us-central1/datasets/7532459492777132032
To use this ImageDataset in another session:
ds = aiplatform.ImageDataset('projects/522977795627/locations/us-central1/datasets/7532459492777132032')
Importing ImageDataset data: projects/522977795627/locations/us-central1/datasets/7532459492777132032
Import ImageDataset data backing LRO: projects/522977795627/locations/us-central1/datasets/7532459492777132032/operations/3010593197698056192
ImageDataset data imported. Resource name: projects/522977795627/locations/us-central1/datasets/7532459492777132032


Create an AutoML training job on this dataset:

**TODO**

# Exercise Solutions

## 1. to 8.

1. A SavedModel contains a TensorFlow model, including its architecture (a computation graph) and its weights. It is stored as a directory containing a _saved_model.pb_ file, which defines the computation graph (represented as a serialized protocol buffer), and a _variables_ subdirectory containing the variable values. For models containing a large number of weights, these variable values may be split across multiple files. A SavedModel also includes an _assets_ subdirectory that may contain additional data, such as vocabulary files, class names, or some example instances for this model. To be more accurate, a SavedModel can contain one or more _metagraphs_. A metagraph is a computation graph plus some function signature definitions (including their input and output names, types, and shapes). Each metagraph is identified by a set of tags. To inspect a SavedModel, you can use the command-line tool `saved_model_cli` or just load it using `tf.saved_model.load()` and inspect it in Python.
2. TF Serving allows you to deploy multiple TensorFlow models (or multiple versions of the same model) and make them accessible to all your applications easily via a REST API or a gRPC API. Using your models directly in your applications would make it harder to deploy a new version of a model across all applications. Implementing your own microservice to wrap a TF model would require extra work, and it would be hard to match TF Serving's features. TF Serving has many features: it can monitor a directory and autodeploy the models that are placed there, and you won't have to change or even restart any of your applications to benefit from the new model versions; it's fast, well tested, and scales very well; and it supports A/B testing of experimental models and deploying a new model version to just a subset of your users (in this case the model is called a _canary_). TF Serving is also capable of grouping individual requests into batches to run them jointly on the GPU. To deploy TF Serving, you can install it from source, but it is much simpler to install it using a Docker image. To deploy a cluster of TF Serving Docker images, you can use an orchestration tool such as Kubernetes, or use a fully hosted solution such as Google Vertex AI.
3. To deploy a model across multiple TF Serving instances, all you need to do is configure these TF Serving instances to monitor the same _models_ directory, and then export your new model as a SavedModel into a subdirectory.
4. The gRPC API is more efficient than the REST API. However, its client libraries are not as widely available, and if you activate compression when using the REST API, you can get almost the same performance. So, the gRPC API is most useful when you need the highest possible performance and the clients are not limited to the REST API.
5. To reduce a model's size so it can run on a mobile or embedded device, TFLite uses several techniques:
    * It provides a converter which can optimize a SavedModel: it shrinks the model and reduces its latency. To do this, it prunes all the operations that are not needed to make predictions (such as training operations), and it optimizes and fuses operations whenever possible.
    * The converter can also perform post-training quantization: this technique dramatically reduces the model’s size, so it’s much faster to download and store.
    * It saves the optimized model using the FlatBuffer format, which can be loaded to RAM directly, without parsing. This reduces the loading time and memory footprint.
6. Quantization-aware training consists in adding fake quantization operations to the model during training. This allows the model to learn to ignore the quantization noise; the final weights will be more robust to quantization.
7. Model parallelism means chopping your model into multiple parts and running them in parallel across multiple devices, hopefully speeding up the model during training or inference. Data parallelism means creating multiple exact replicas of your model and deploying them across multiple devices. At each iteration during training, each replica is given a different batch of data, and it computes the gradients of the loss with regard to the model parameters. In synchronous data parallelism, the gradients from all replicas are then aggregated and the optimizer performs a Gradient Descent step. The parameters may be centralized (e.g., on parameter servers) or replicated across all replicas and kept in sync using AllReduce. In asynchronous data parallelism, the parameters are centralized and the replicas run independently from each other, each updating the central parameters directly at the end of each training iteration, without having to wait for the other replicas. To speed up training, data parallelism turns out to work better than model parallelism, in general. This is mostly because it requires less communication across devices. Moreover, it is much easier to implement, and it works the same way for any model, whereas model parallelism requires analyzing the model to determine the best way to chop it into pieces. That said, research in this domain is making quick progress (e.g., PipeDream or Pathways), so a mix of model parallelism and data parallelism is probably the way forward.
8. When training a model across multiple servers, you can use the following distribution strategies:
    * The `MultiWorkerMirroredStrategy` performs mirrored data parallelism. The model is replicated across all available servers and devices, and each replica gets a different batch of data at each training iteration and computes its own gradients. The mean of the gradients is computed and shared across all replicas using a distributed AllReduce implementation (NCCL by default), and all replicas perform the same Gradient Descent step. This strategy is the simplest to use since all servers and devices are treated in exactly the same way, and it performs fairly well. In general, you should use this strategy. Its main limitation is that it requires the model to fit in RAM on every replica.
    * The `ParameterServerStrategy` performs asynchronous data parallelism. The model is replicated across all devices on all workers, and the parameters are sharded across all parameter servers. Each worker has its own training loop, running asynchronously with the other workers; at each training iteration, each worker gets its own batch of data and fetches the latest version of the model parameters from the parameter servers, then it computes the gradients of the loss with regard to these parameters, and it sends them to the parameter servers. Lastly, the parameter servers perform a Gradient Descent step using these gradients. This strategy is generally slower than the previous strategy, and a bit harder to deploy, since it requires managing parameter servers. However, it can be useful in some situations, especially when you can take advantage of the asynchronous updates, for example to reduce I/O bottlenecks. This depends on many factors, including hardware, network topology, number of servers, model size, and more, so your mileage may vary.

## 9.
_Exercise: Train a model (any model you like) and deploy it to TF Serving or Google Vertex AI. Write the client code to query it using the REST API or the gRPC API. Update the model and deploy the new version. Your client code will now query the new version. Roll back to the first version._

Please follow the steps in the <a href="#Deploying-TensorFlow-models-to-TensorFlow-Serving-(TFS)">Deploying TensorFlow models to TensorFlow Serving</a> section above.

# 10.
_Exercise: Train any model across multiple GPUs on the same machine using the `MirroredStrategy` (if you do not have access to GPUs, you can use Colaboratory with a GPU Runtime and create two virtual GPUs). Train the model again using the `CentralStorageStrategy `and compare the training time._

Please follow the steps in the [Distributed Training](#Distributed-Training) section above.

# 11.
_Exercise: Train a small model on Google Vertex AI, using TensorFlow Cloud Tuner for hyperparameter tuning._

Please follow the instructions in the _Hyperparameter Tuning using TensorFlow Cloud Tuner_ section in the book.

# Congratulations!

You've reached the end of the book! I hope you found it useful. 😊